# Qwen3-4B-Thinking-2507 — evaluate the ViNumQA SFT adapter

Loads the LoRA adapter produced by `qwen3-4b-thinking-2507-stf-w-reasoning-trace.ipynb` and
scores Program Accuracy / Execution Accuracy on `test.json`, using the same parser as every
other notebook in this repo so the numbers stay comparable.

Split out from the training notebook because the two together exceed Kaggle's 12h session
limit. Before running: attach the training notebook's output as a data source and point
`ADAPTER_DIR` below at it.

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups (incl. Kaggle)
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install -q tabulate
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Load the fine-tuned adapter

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 8192

# Path to the adapter produced by the training run. Check the Data panel for
# the exact folder name -- if you saved the training run's output as a Kaggle
# Dataset, the path looks like
#   /kaggle/input/<dataset-slug>/qwen3-4b-thinking-2507-vinumqa-sft-adapter
# The folder must be the *-adapter one (containing adapter_model.safetensors),
# not the *-sft one, which holds mid-training checkpoints.
ADAPTER_DIR = "/kaggle/input/<your-dataset-slug>/qwen3-4b-thinking-2507-vinumqa-sft-adapter"

import os
assert os.path.isdir(ADAPTER_DIR), (
    f"{ADAPTER_DIR} not found -- set ADAPTER_DIR to the folder shown in the Data panel."
)
print("Found adapter:", sorted(os.listdir(ADAPTER_DIR))[:6])

# from_pretrained on an adapter directory pulls the base model automatically and
# applies the LoRA weights on top.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_DIR,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
print("Adapter loaded.")

### Data + prompt (identical to the training notebook)

In [ ]:
import pandas as pd
from tabulate import tabulate

test_df = pd.read_json('/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa/test.json')
print(f"test={len(test_df)}")

In [ ]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_reasoning_trace(sample):
    # None for samples without a verified trace -- kept as None (not "")
    # so build_conversation can tell the two cases apart.
    return sample["qa"].get("reasoning_trace")

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

def process_split(df):
    df = df.copy()
    df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
    df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
    df["table_processed"] = df.apply(formatting_table, axis=1)
    df["input_question"] = df.apply(processing_input_question, axis=1)
    df["program_processed"] = df.apply(processing_program_content, axis=1)
    df["answer_processed"] = df.apply(processing_answer_content, axis=1)
    df = df[["pre_text_processed", "table_processed", "post_text_processed", "input_question",
             "program_processed", "answer_processed"]]
    df.columns = ["pre_text", "table", "post_text", "question", "program", "answer"]
    return df

test_df = process_split(test_df)
test_df["generated_program"] = ""
test_df.head(3)

In [ ]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(val1, val2, val3, ...) -> sum of the values
8. table_average(val1, val2, val3, ...) -> arithmetic mean of the values
9. table_max(val1, val2, val3, ...) -> maximum of the values
10. table_min(val1, val2, val3, ...) -> minimum of the values

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- Do not use square brackets `[]` inside table-type functions (e.g. write table_max(1, 2, 3), not table_max([1, 2, 3])).
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use \'none\'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""

# Same prompt format as the 0-shot/1-shot/sft notebooks, so results are
# comparable across all experiments (only the training regime differs).

### PA / EA

In [ ]:
"""
Parser + evaluation utilities for FinQA/VLSP-2025 style computation programs.
(Same implementation as the 0-shot/1-shot/sft notebooks.)
"""

import re
from typing import List, Tuple, Union, Optional

VALID_OPERATORS = {
    "add", "subtract", "multiply", "divide", "exp", "greater",
    "table_sum", "table_average", "table_max", "table_min",
}
_BINARY_OPS = {"add", "subtract", "multiply", "divide", "exp", "greater"}
_TABLE_OPS = {"table_sum", "table_average", "table_max", "table_min"}
_NUM_RE = re.compile(r"^-?\d+(\.\d+)?$")
_REF_RE = re.compile(r"^#(\d+)$")


def _parse_numeric_literal(raw: str) -> float:
    cleaned = raw.replace(",", "").replace("$", "").strip()
    is_percent = cleaned.endswith("%")
    if is_percent:
        cleaned = cleaned[:-1].strip()
    if not _NUM_RE.match(cleaned):
        raise ValueError(f"Cannot parse numeric argument: '{raw}'")
    value = float(cleaned)
    return value / 100.0 if is_percent else value


def extract_program(raw_text: str) -> str:
    text = raw_text.strip()
    text = re.sub(r"```[a-zA-Z]*", "", text)
    text = text.replace("```", "").strip()
    op_names = "|".join(sorted(VALID_OPERATORS, key=len, reverse=True))
    pattern = rf"\b(?:{op_names})\([^()]*\)"
    matches = re.findall(pattern, text)
    if matches:
        return ", ".join(m.strip() for m in matches)
    return text


def _split_top_level(s: str, sep: str = ",") -> List[str]:
    parts, depth, current = [], 0, []
    for ch in s:
        if ch == "(":
            depth += 1; current.append(ch)
        elif ch == ")":
            depth -= 1; current.append(ch)
        elif ch == sep and depth == 0:
            parts.append("".join(current)); current = []
        else:
            current.append(ch)
    if current:
        parts.append("".join(current))
    return [p.strip() for p in parts if p.strip() != ""]


def parse_program(program_str: str) -> List[Tuple[str, List[str]]]:
    program_str = program_str.strip().rstrip(",").strip()
    if not program_str:
        raise ValueError("Empty program string.")
    steps: List[Tuple[str, List[str]]] = []
    call_pattern = re.compile(r"\s*([a-zA-Z_]+)\(([^()]*)\)\s*")
    pos = 0
    text = program_str
    while pos < len(text):
        m = call_pattern.match(text, pos)
        if not m:
            raise ValueError(f"Malformed program near: '{text[pos:pos+30]}...'")
        op = m.group(1).strip()
        if op not in VALID_OPERATORS:
            raise ValueError(f"Unknown operator: '{op}'")
        args = _split_top_level(m.group(2), sep=",")
        steps.append((op, args))
        pos = m.end()
        if pos < len(text) and text[pos] == ",":
            pos += 1
    if not steps:
        raise ValueError("No valid steps parsed.")
    return steps


def _resolve_arg(arg: str, results: List[float]) -> Optional[float]:
    arg = arg.strip()
    ref_match = _REF_RE.match(arg)
    if ref_match:
        idx = int(ref_match.group(1))
        if idx >= len(results):
            raise ValueError(f"Reference #{idx} used before step {idx} was computed.")
        return results[idx]
    if arg.lower() == "none":
        return None
    return _parse_numeric_literal(arg)


def execute_program(program_str: str) -> Tuple[float, List[float]]:
    steps = parse_program(program_str)
    results: List[float] = []
    for op, raw_args in steps:
        vals = [_resolve_arg(a, results) for a in raw_args]
        if op in _BINARY_OPS:
            if len(vals) != 2:
                raise ValueError(f"Operator '{op}' expects 2 args, got {len(vals)}.")
            a, b = vals
            if a is None or b is None:
                raise ValueError(f"Operator '{op}' received a 'none' operand.")
            if op == "add": r = a + b
            elif op == "subtract": r = a - b
            elif op == "multiply": r = a * b
            elif op == "divide":
                if b == 0:
                    raise ValueError("Division by zero.")
                r = a / b
            elif op == "exp": r = a ** b
            elif op == "greater": r = 1.0 if a > b else 0.0
        elif op in _TABLE_OPS:
            if len(vals) < 1:
                raise ValueError(f"Operator '{op}' requires at least 1 arg.")
            if any(v is None for v in vals):
                raise ValueError(f"Operator '{op}' received a 'none' operand.")
            if op == "table_sum": r = sum(vals)
            elif op == "table_average": r = sum(vals) / len(vals)
            elif op == "table_max": r = max(vals)
            elif op == "table_min": r = min(vals)
        else:
            raise ValueError(f"Unknown operator: '{op}'")
        results.append(r)
    return results[-1], results


def _normalize_program(program_str: str) -> List[Tuple[str, Tuple[str, ...]]]:
    steps = parse_program(program_str)
    normalized = []
    for op, args in steps:
        norm_args = []
        for a in args:
            a = a.strip()
            if _REF_RE.match(a) or a.lower() == "none":
                norm_args.append(a.lower())
            else:
                try:
                    norm_args.append(f"{round(_parse_numeric_literal(a), 6)}")
                except ValueError:
                    norm_args.append(a)
        normalized.append((op, tuple(norm_args)))
    return normalized


def compute_program_accuracy(generated_program, gold_program, alternative_gold_programs=None) -> float:
    try:
        gen_norm = _normalize_program(generated_program)
    except ValueError:
        return 0.0
    gold_candidates = [gold_program] + (alternative_gold_programs or [])
    for gold in gold_candidates:
        try:
            gold_norm = _normalize_program(gold)
        except ValueError:
            continue
        if gen_norm == gold_norm:
            return 1.0
    return 0.0


def compute_execution_accuracy(generated_program, gold_answer, rel_tol: float = 1e-3, abs_tol: float = 1e-4) -> float:
    try:
        predicted, _ = execute_program(generated_program)
    except (ValueError, ZeroDivisionError, OverflowError):
        return 0.0
    try:
        gold = _parse_numeric_literal(str(gold_answer))
    except ValueError:
        return 0.0
    if abs(predicted - gold) <= max(abs_tol, rel_tol * abs(gold)):
        return 1.0
    return 0.0


def evaluate_dataframe(df, generated_col="generated_program", gold_program_col="program",
                        gold_answer_col="answer", extract_first=True):
    df = df.copy()
    pa_scores, ea_scores = [], []
    for _, row in df.iterrows():
        raw_generated = row[generated_col]
        generated = extract_program(raw_generated) if extract_first else raw_generated
        pa = compute_program_accuracy(generated, row[gold_program_col])
        ea = compute_execution_accuracy(generated, row[gold_answer_col])
        pa_scores.append(pa)
        ea_scores.append(ea)
    df["pa_score"] = pa_scores
    df["ea_score"] = ea_scores
    summary = {
        "program_accuracy": sum(pa_scores) / len(pa_scores) if pa_scores else 0.0,
        "execution_accuracy": sum(ea_scores) / len(ea_scores) if ea_scores else 0.0,
    }
    return df, summary

In [ ]:
import gc
from pathlib import Path
from tqdm import tqdm

QWEN3_THINK_END_TOKEN_ID = 151668  # "</think>"

# One prompt at a time, deliberately. Batching several prompts together requires
# left padding, and Unsloth's fast inference path derives token positions from
# the cache length rather than from the attention mask -- so padded rows decode
# at the wrong positions. Measured on the wo-reasoning model, that cost 0.6419
# -> 0.6338 PA. Extra votes come from num_return_sequences instead: those share
# a single prompt, so there is nothing to pad, and memory stays bounded by
# N_VOTES rather than by N_VOTES x batch.
#
# N_VOTES = 1 is the like-for-like number. Raising it multiplies the runtime,
# and this model emits a full reasoning trace per sample (~24s each on a T4,
# so ~3.3h for one pass over the 497 test questions) -- k=5 would not fit a
# session. Run k=1 first; the checkpoint below lets a later pass resume.
N_VOTES = 1
# 2048, matching the run this adapter came from. Too low a cap is not a mild
# loss: if generation is cut off before the model closes </think>, strip_think
# finds no closing tag and hands the whole reasoning trace to the parser as if
# it were the program, so a sample that would have been right scores 0 on both
# metrics. Generation stops at EOS anyway, so the higher cap costs nothing on
# samples that finish early.
EVAL_MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.6   # only used when N_VOTES > 1; Qwen3 thinking defaults
TOP_P = 0.95

CKPT_PATH = Path(f"/kaggle/working/eval_partial_k{N_VOTES}.csv")
if CKPT_PATH.exists():
    _done = pd.read_csv(CKPT_PATH, index_col=0).fillna("")
    test_df.loc[_done.index, "generated_program"] = _done["generated_program"].values
    print(f"Resumed {(test_df['generated_program'] != '').sum()} / {len(test_df)} from checkpoint.")


def build_prompt(row):
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"],
        post_text=row["post_text"], question=row["question"],
    )
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_msg},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )


def strip_think(output_ids):
    """Keep only what follows the final </think>; the model always emits one."""
    ids = list(output_ids)
    try:
        cut = len(ids) - ids[::-1].index(QWEN3_THINK_END_TOKEN_ID)
    except ValueError:
        cut = 0
    return tokenizer.decode(ids[cut:], skip_special_tokens=True).strip()


def vote(candidates):
    """Most common program among candidates, keyed by normalised form so that
    cosmetically different but structurally identical programs share a vote."""
    cleaned = [extract_program(c) for c in candidates if c and c.strip()]
    if not cleaned:
        return ""
    keyed = {}
    for c in cleaned:
        try:
            key = str(_normalize_program(c))
        except Exception:
            key = c.strip()
        keyed.setdefault(key, []).append(c)
    best = max(keyed, key=lambda k: (len(keyed[k]), -cleaned.index(keyed[k][0])))
    return keyed[best][0]


n_unclosed = 0
todo = [i for i in test_df.index if not str(test_df.at[i, "generated_program"]).strip()]

for n, df_index in enumerate(tqdm(todo, desc=f"Generating (k={N_VOTES})")):
    enc = tokenizer([build_prompt(test_df.loc[df_index])], return_tensors="pt").to(model.device)

    gen_kwargs = dict(max_new_tokens=EVAL_MAX_NEW_TOKENS)
    if N_VOTES > 1:
        gen_kwargs.update(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
                          num_return_sequences=N_VOTES)

    try:
        with torch.no_grad():
            out = model.generate(**enc, **gen_kwargs)
        prompt_len = enc["input_ids"].shape[1]
        gen_only = [r[prompt_len:].tolist() for r in out]
        # A generation that never closed </think> was almost certainly cut off
        # by the token cap; count them so a too-low cap is visible rather than
        # silently scoring zeros.
        n_unclosed += sum(1 for g in gen_only if QWEN3_THINK_END_TOKEN_ID not in g)
        cands = [strip_think(g) for g in gen_only]
        test_df.at[df_index, "generated_program"] = vote(cands) if N_VOTES > 1 else cands[0]
        del enc, out
    except torch.cuda.OutOfMemoryError:
        print(f"  OOM on index {df_index}; leaving it blank (scored 0).")
        test_df.at[df_index, "generated_program"] = ""

    if n % 25 == 0:
        test_df[["generated_program"]].to_csv(CKPT_PATH)
        gc.collect()
        torch.cuda.empty_cache()

test_df[["generated_program"]].to_csv(CKPT_PATH)
print(f"Done. {(test_df['generated_program'] != '').sum()} / {len(test_df)} generated "
      f"(N_VOTES={N_VOTES}).")
print(f"Generations that never closed </think>: {n_unclosed} "
      f"-- these were cut off by EVAL_MAX_NEW_TOKENS and score 0; "
      f"raise the cap if this is not near zero.")

In [ ]:
df_scored, summary = evaluate_dataframe(test_df)
print(summary)  # {\'program_accuracy\': ..., \'execution_accuracy\': ...}